# Best Stage 1-3 Recipe: AE-Pretrained CNN + Trainable Text Projection

This notebook is the streamlined reproduction path for the known-best ALE3DCNN contrastive recipe: `ae_pretrained_finetune_cnn_pretrained_text_trainable`.

It runs only the stages needed for the best retrieval model:

1. Stage 1: pretrain an atlas-free ALE3DCNN autoencoder.
2. Stage 2: select the best autoencoder checkpoint for encoder initialization.
3. Stage 3: train the contrastive ALE3DCNN model with the CNN encoder fine-tuned and the pretrained text projection left trainable.

There is intentionally no Stage 4 text-to-brain generation in this notebook.


## 1. Runtime, Dependencies, and Repo


In [ ]:
import json
import os
import platform
import shlex
import subprocess
import sys
import time
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch check failed:', exc)

RUN_IN_COLAB = 'google.colab' in sys.modules
INSTALL_DEPS = os.environ.get('NEUROVLM_INSTALL_DEPS', '1') != '0'
if INSTALL_DEPS:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'nilearn', 'nibabel', 'huggingface-hub', 'safetensors',
        'adapters', 'transformers', 'pyarrow', 'matplotlib',
        'pandas', 'scikit-learn', 'tqdm', 'umap-learn', 'pyyaml',
    ])

REPO_URL = os.environ.get('NEUROVLM_REPO_URL', 'https://github.com/neurovlm/neurovlm.git')
REPO_BRANCH = os.environ.get('NEUROVLM_REPO_BRANCH', 'neurovlm_gnn')

def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'experiments' / '3dcnn').exists():
            return candidate
    return start

if RUN_IN_COLAB:
    repo_dir = Path(os.environ.get('NEUROVLM_REPO_DIR', '/content/neurovlm_gnn'))
    if not repo_dir.exists():
        subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(repo_dir)])
    else:
        subprocess.check_call(['git', '-C', str(repo_dir), 'fetch', 'origin', REPO_BRANCH])
        subprocess.check_call(['git', '-C', str(repo_dir), 'checkout', REPO_BRANCH])
        subprocess.check_call(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', REPO_BRANCH])
else:
    repo_dir = find_repo_root(os.environ.get('NEUROVLM_REPO_DIR', Path.cwd()))

REPO_DIR = repo_dir.resolve()
os.chdir(REPO_DIR)
if INSTALL_DEPS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[viz,notebook,metrics]'])

for path in [REPO_DIR / 'experiments' / '3dcnn', REPO_DIR / 'src']:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print('Working directory:', REPO_DIR)


## 2. Drive and Run Configuration


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

RUN_STAMP = time.strftime('%Y%m%d_%H%M%S')
DRIVE_ROOT = Path(os.environ.get(
    'NEUROVLM_DRIVE_ROOT',
    '/content/drive/MyDrive/neurovlm' if RUN_IN_COLAB else str(REPO_DIR / 'runs'),
)).expanduser()
RUN_ROOT = DRIVE_ROOT / 'runs_ale_3dcnn_best_stage1_stage3' / f'best_stage1_stage3_{RUN_STAMP}'
LOG_DIR = RUN_ROOT / 'logs'
AE_RUN_DIR = RUN_ROOT / '01_stage1_autoencoder_pretraining'
STAGE3_RUN_DIR = RUN_ROOT / '03_stage3_ae_pretrained_finetune_cnn_pretrained_text_trainable'

LOCAL_CACHE_DIR = Path(os.environ.get('ALE_CNN_LOCAL_CACHE_DIR', '/content/ale_caches_ale_3dcnn' if RUN_IN_COLAB else str(REPO_DIR / 'cache' / 'ale_caches'))).expanduser()
DRIVE_CACHE_DIR = DRIVE_ROOT / 'data_ale_3dcnn' / 'ale_caches'
EVAL_RESOURCE_DIR = Path(os.environ.get(
    'NEUROVLM_EVAL_RESOURCE_DIR',
    '/content/drive/MyDrive/neurovlm_evaluation_resources' if RUN_IN_COLAB else str(REPO_DIR / 'experiments' / 'evaluation_resources'),
)).expanduser()

AE_TRAINER = REPO_DIR / 'experiments' / '3dcnn' / 'atlas_free_cnn' / 'training' / 'train_ale_cnn_autoencoder.py'
STAGE3_TRAINER = REPO_DIR / 'experiments' / '3dcnn' / 'atlas_free_cnn' / 'training' / 'train_ale_cnn.py'

for directory in [RUN_ROOT, LOG_DIR, AE_RUN_DIR, STAGE3_RUN_DIR, LOCAL_CACHE_DIR, DRIVE_CACHE_DIR, EVAL_RESOURCE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RESOLUTION_MM = 4.0
KERNEL_FWHM_MM = 9.0
CACHE_DTYPE = 'float16'
CACHE_FILE = LOCAL_CACHE_DIR / f'atlas_free_ale_{int(RESOLUTION_MM)}mm_fwhm{str(KERNEL_FWHM_MM).replace(".", "p")}_crop_{CACHE_DTYPE}.pt'

# Best historical plain-CNN recipe. Override with env vars for quick smoke tests.
AE_EPOCHS = int(os.environ.get('ALE_CNN_AE_EPOCHS', '150'))
CONTRASTIVE_EPOCHS = int(os.environ.get('ALE_CNN_CONTRASTIVE_EPOCHS', '200'))
AE_BATCH_CANDIDATES = os.environ.get('ALE_CNN_AE_BATCH_CANDIDATES', '256,192,128,96,64,32,16,8,4')
CONTRASTIVE_BATCH_CANDIDATES = os.environ.get('ALE_CNN_BATCH_CANDIDATES', '2048,1536,1024,768,512,384,256,192,128,96,64,32,16,8,4')
RUN_SEMANTIC_EVAL = os.environ.get('ALE_CNN_SEMANTIC_EVAL', '1') != '0'
SKIP_COMPLETED = os.environ.get('ALE_CNN_SKIP_COMPLETED', '1') != '0'

config = {
    'recipe': 'ae_pretrained_finetune_cnn_pretrained_text_trainable',
    'run_root': str(RUN_ROOT),
    'ae_run_dir': str(AE_RUN_DIR),
    'stage3_run_dir': str(STAGE3_RUN_DIR),
    'cache_file': str(CACHE_FILE),
    'eval_resource_dir': str(EVAL_RESOURCE_DIR),
    'ae_epochs': AE_EPOCHS,
    'contrastive_epochs': CONTRASTIVE_EPOCHS,
    'run_semantic_eval': RUN_SEMANTIC_EVAL,
}
with (RUN_ROOT / 'best_stage1_stage3_config.json').open('w') as f:
    json.dump(config, f, indent=2)

print(json.dumps(config, indent=2))


## 3. Command Helper


In [ ]:
def run_command(args, log_file):
    args = [str(arg) for arg in args]
    log_file = Path(log_file)
    log_file.parent.mkdir(parents=True, exist_ok=True)
    command_text = ' '.join(shlex.quote(arg) for arg in args)
    print(command_text)
    with log_file.open('w') as f:
        f.write(command_text + '\n\n')
        proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            f.write(line)
        code = proc.wait()
        f.write(f'\nEXIT_CODE={code}\n')
    if code != 0:
        tail = log_file.read_text(errors='replace').splitlines()[-80:]
        print('--- log tail ---')
        print('\n'.join(tail))
        print('--- end log tail ---')
        raise RuntimeError(f'Command failed with exit code {code}: {log_file}')
    return code

def existing_checkpoint(path):
    path = Path(path)
    return path.exists() and path.is_file() and path.stat().st_size > 0


## 4. Stage 1: Pretrain Atlas-Free ALE3DCNN Autoencoder


In [ ]:
ae_checkpoint_dir = AE_RUN_DIR / 'checkpoints'
ae_checkpoint_dir.mkdir(parents=True, exist_ok=True)
ae_best_checkpoint = ae_checkpoint_dir / 'best_cnn_autoencoder.pt'

ae_cmd = [
    sys.executable, AE_TRAINER,
    '--mode', 'atlas_free',
    '--model', 'ale_3dcnn',
    '--epochs', AE_EPOCHS,
    '--batch-size-auto',
    '--batch-size-candidates', AE_BATCH_CANDIDATES,
    '--lr', '3e-4',
    '--weight-decay', '1e-4',
    '--val-interval', '5',
    '--early-stopping-patience', '20',
    '--base-channels', '48',
    '--num-blocks', '4',
    '--latent-dim', '384',
    '--dropout', '0.1',
    '--norm', 'group',
    '--pooling', 'max',
    '--kernel-fwhm-mm', str(KERNEL_FWHM_MM),
    '--resolution-mm', str(RESOLUTION_MM),
    '--cache-dtype', CACHE_DTYPE,
    '--cache-file', CACHE_FILE,
    '--lambda-recon', '1.0',
    '--lambda-dice', '0.5',
    '--lambda-topk', '0.5',
    '--lambda-corr', '0.25',
    '--recon-alpha', '10.0',
    '--recon-gamma', '1.0',
    '--prediction-activation', 'none',
    '--num-workers', '0',
    '--run-dir', AE_RUN_DIR,
    '--checkpoint-dir', ae_checkpoint_dir,
]

if SKIP_COMPLETED and existing_checkpoint(ae_best_checkpoint):
    print('Skipping completed Stage 1:', ae_best_checkpoint)
else:
    run_command(ae_cmd, LOG_DIR / '01_stage1_autoencoder_pretraining.log')

print('Stage 1 checkpoint:', ae_best_checkpoint)


## 5. Stage 2: Select Autoencoder Checkpoint


In [ ]:
checkpoint_candidates = [
    ae_checkpoint_dir / 'best_cnn_autoencoder.pt',
    ae_checkpoint_dir / 'best_val_mse.pt',
    ae_checkpoint_dir / 'last_cnn_autoencoder.pt',
]
BEST_AE_CHECKPOINT = next((path for path in checkpoint_candidates if existing_checkpoint(path)), None)
if BEST_AE_CHECKPOINT is None:
    raise FileNotFoundError(f'No usable autoencoder checkpoint found in {ae_checkpoint_dir}')

stage2_selection = {
    'selected_checkpoint': str(BEST_AE_CHECKPOINT),
    'selection_order': [str(path) for path in checkpoint_candidates],
    'recipe': 'load AE encoder, fine-tune CNN, initialize pretrained text projection, keep text projection trainable',
}
with (RUN_ROOT / '02_stage2_selected_autoencoder_checkpoint.json').open('w') as f:
    json.dump(stage2_selection, f, indent=2)

print(json.dumps(stage2_selection, indent=2))


## 6. Stage 3: Train Best Contrastive Variant


In [ ]:
stage3_checkpoint_dir = STAGE3_RUN_DIR / 'checkpoints'
stage3_checkpoint_dir.mkdir(parents=True, exist_ok=True)
stage3_best_checkpoint = stage3_checkpoint_dir / 'best_ale_cnn.pt'

stage3_cmd = [
    sys.executable, STAGE3_TRAINER,
    '--mode', 'atlas_free',
    '--model', 'ale_3dcnn',
    '--epochs', CONTRASTIVE_EPOCHS,
    '--batch-size-auto',
    '--batch-size-candidates', CONTRASTIVE_BATCH_CANDIDATES,
    '--lr-cnn', '1e-4',
    '--lr-proj', '1e-5',
    '--warmup-epochs', '5',
    '--temperature', '0.07',
    '--val-interval', '5',
    '--early-stopping-patience', '25',
    '--base-channels', '48',
    '--num-blocks', '4',
    '--out-dim', '384',
    '--dropout', '0.1',
    '--norm', 'group',
    '--pooling', 'max',
    '--kernel-fwhm-mm', str(KERNEL_FWHM_MM),
    '--resolution-mm', str(RESOLUTION_MM),
    '--cache-dtype', CACHE_DTYPE,
    '--cache-file', CACHE_FILE,
    '--encoder-init', 'autoencoder_pretrained',
    '--autoencoder-checkpoint', BEST_AE_CHECKPOINT,
    '--ae-init-variant', 'stage1_autoencoder_pretrained',
    '--ae-checkpoint-selection', 'best_val_loss',
    '--text-proj-init', 'pretrained_infonce',
    '--train-sanity-n', '512',
    '--num-workers', '0',
    '--run-dir', STAGE3_RUN_DIR,
    '--checkpoint-dir', stage3_checkpoint_dir,
    '--comparison-file', RUN_ROOT / 'stage3_best_recipe_comparison.csv',
]

if RUN_SEMANTIC_EVAL:
    stage3_cmd.extend(['--semantic-eval', '--eval-resource-dir', EVAL_RESOURCE_DIR])

if SKIP_COMPLETED and existing_checkpoint(stage3_best_checkpoint):
    print('Skipping completed Stage 3:', stage3_best_checkpoint)
else:
    run_command(stage3_cmd, LOG_DIR / '03_stage3_best_contrastive_variant.log')

print('Stage 3 run:', STAGE3_RUN_DIR)


## 7. Output Summary


In [ ]:
summary = {
    'recipe': 'ae_pretrained_finetune_cnn_pretrained_text_trainable',
    'stage1_autoencoder_run_dir': str(AE_RUN_DIR),
    'stage2_selected_autoencoder_checkpoint': str(BEST_AE_CHECKPOINT),
    'stage3_contrastive_run_dir': str(STAGE3_RUN_DIR),
    'stage3_checkpoints_dir': str(stage3_checkpoint_dir),
    'stage4_included': False,
}
summary_path = RUN_ROOT / 'best_stage1_stage3_summary.json'
with summary_path.open('w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print('Saved summary:', summary_path)
